# Setup

In [ ]:
%%capture
import os

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -q -r requirements.txt
else:
    !pip install -q --upgrade pip
    !pip install -q -r requirements.txt

In [ ]:
import random
import numpy as np
import torch

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
from pathlib import Path
import sys
from torch.utils.data import DataLoader

root = Path.cwd()
sys.path.append(str(root))

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Config

In [ ]:
from src.config import Paths, EnvCfg, DataCfg, TrainCfg, PPOCfg

paths = Paths()
env_cfg = EnvCfg()
data_cfg = DataCfg()
train_cfg = TrainCfg()
ppo_cfg = PPOCfg()

paths.mkdirs()

# Env

In [ ]:
from src.envs import build_env

env = build_env(size=env_cfg.size, slippery=env_cfg.slippery)

In [ ]:
obs, _ = env.reset()
obs

# Expert Data

In [ ]:
from src.expert import collect_expert_trajectories

records = collect_expert_trajectories(
    env,
    n_traj=data_cfg.n_traj,
    seed=data_cfg.seed,
)

In [ ]:
len(records)

# Split

In [ ]:
from src.data import split_records

train_records, val_records = split_records(
    records,
    val_split=data_cfg.val_split,
    seed=data_cfg.seed,
)

# Datasets

In [ ]:
from src.data import ActionOnlyDataset, ActionReasonDataset, TinyTokenizer, collate_reason

tok = TinyTokenizer()

In [ ]:
ds_action_train = ActionOnlyDataset(train_records)
ds_action_val = ActionOnlyDataset(val_records)

In [ ]:
ds_reason_train = ActionReasonDataset(train_records, tok)
ds_reason_val = ActionReasonDataset(val_records, tok)

In [ ]:
dl_action_train = DataLoader(ds_action_train, batch_size=train_cfg.batch_size, shuffle=True)
dl_action_val = DataLoader(ds_action_val, batch_size=train_cfg.batch_size, shuffle=False)

In [ ]:
dl_reason_train = DataLoader(ds_reason_train, batch_size=train_cfg.batch_size, shuffle=True, collate_fn=collate_reason)
dl_reason_val = DataLoader(ds_reason_val, batch_size=train_cfg.batch_size, shuffle=False, collate_fn=collate_reason)

# SFT Action

In [ ]:
from src.model import NanoVLMAction
from src.trainers import run_sft_action

model_action = NanoVLMAction(n_actions=4)

In [ ]:
hist_action, best_action_acc = run_sft_action(
    model_action,
    dl_action_train,
    dl_action_val,
    train_cfg,
)
best_action_acc

# SFT Reason

In [ ]:
from src.model import NanoVLMReason
from src.trainers import run_sft_reason

model_reason = NanoVLMReason(vocab_size=len(tok), n_actions=4)

In [ ]:
hist_reason = run_sft_reason(
    model_reason,
    dl_reason_train,
    dl_reason_val,
    train_cfg,
)
hist_reason[-1]

# PPO Action

In [ ]:
from src.trainers import ppo_update_step

model_action = model_action.to(device)
opt = torch.optim.AdamW(model_action.parameters(), lr=ppo_cfg.lr)

In [ ]:
batch = next(iter(dl_action_train))
images = batch['image'].to(device)
acts = batch['action'].to(device)

In [ ]:
with torch.no_grad():
    old_logits = model_action(images)
    old_dist = torch.distributions.Categorical(logits=old_logits)
    old_logp = old_dist.log_prob(acts)

adv = torch.ones_like(old_logp)

In [ ]:
for _ in range(ppo_cfg.total_updates):
    for _ in range(ppo_cfg.update_epochs):
        ppo_update_step(
            model_action,
            {'image': images, 'action': acts},
            old_logp,
            adv,
            ppo_cfg,
            opt,
        )

# Eval

In [ ]:
from src.envs import obs_to_image
import numpy as np

In [ ]:
model_action.eval()

In [ ]:
def run_episode():
    obs, _ = env.reset()
    done = False
    total = 0.0
    while not done:
        img = obs_to_image(env)
        x = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0) / 255.0
        x = x.to(device)
        with torch.no_grad():
            a = model_action(x).argmax(dim=-1).item()
        obs, r, term, trunc, _ = env.step(a)
        total += float(r)
        done = term or trunc
    return total

In [ ]:
scores = [run_episode() for _ in range(200)]

In [ ]:
float(np.mean(scores))